In [0]:
%run ./utils

# Imports

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Load

In [0]:
dbutils.widgets.text("bronze_table_name", "tb_atendentes")
bronze_table_name = dbutils.widgets.get("bronze_table_name")

df_atendentes = (
    spark.table(f"{bronze_schema}.{bronze_table_name}")
        .dropDuplicates()
        .orderBy("id_atendente")
)

display(df_atendentes.limit(5))

In [0]:
# Definição da janela agrupada pela chave primária e ordenada pelo timestamp decrescente
window_spec = Window.partitionBy("id_atendente").orderBy(F.col("ingestion_datetime").desc())

df_atendentes = (
    df_atendentes
        .withColumn("rn", F.row_number().over(window_spec))
        .filter(F.col("rn") == 1)
        .drop("rn")
        .select(
            F.col("id_atendente").cast("string"),
            "nome_atendente",
            "nivel_atendimento",
            "data_admissao",
            "data_desligamento",
            "status_atendente"
        )
)

# DF final deve ter 25 linhas
print(f"Total de linhas da tabela de atendentes: {df_atendentes.count()}")

display(df_atendentes.limit(5))

# Tratamento do nível de atendimento

In [0]:
df_niveis = (
    df_atendentes
        .select("nivel_atendimento")
        .distinct()
)

display(df_niveis)

In [0]:
nivel_1_texts = ["1", "Nível 1"]
nivel_2_texts = ["2", "Nível 2", "N2"]

df_atendentes = (
    df_atendentes
        .withColumn(
            "nivel_atendimento",
            F.when(F.col("nivel_atendimento").isin(nivel_1_texts), "1")
                .when(F.col("nivel_atendimento").isin(nivel_2_texts), "2")
                .otherwise(F.col("nivel_atendimento"))
        )
)

dq_check(
    "df_atendentes", 
    "Níveis de atendimento devem estar padronizados", 
    df_atendentes, 
    F.col("nivel_atendimento").isin(["1", "2"])
)

# Verificações de qualidade

Não verificamos se o id do atendente é único, pois um mesmo atendente pode vir a trabalhar em dois níveis diferentes.

In [0]:
dq_check(
    "df_atendentes", 
    "Data de desligamento deve ser após data de admissão", 
    df_atendentes, 
    F.col("data_desligamento").isNull() | (F.col("data_desligamento") > F.col("data_admissao"))
)

In [0]:
dq_check(
    "df_atendentes", 
    "Atendente deve estar inativo apenas se tiver sido desligado", 
    df_atendentes, 
    (F.col("data_desligamento").isNull() & (F.col("status_atendente") == "ativo"))
    | (F.col("data_desligamento").isNotNull() & (F.col("status_atendente") == "inativo"))
)

# Salvamento

In [0]:
dbutils.widgets.text("silver_table_name", "tb_atendentes")
silver_table_name = dbutils.widgets.get("silver_table_name")

save_df(df_atendentes, silver_table_name)